In [2]:
!pip install requests pandas tqdm --quiet

import re
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime, timedelta, timezone

# =========================================================
# CONFIG
# =========================================================
SUBREDDIT = "askscience"
START_DATE = "2025-10-06"
END_DATE   = "2025-12-28"

# Week-balanced collection
TARGET_CLEAN_PER_WEEK = 250   # 12 weeks -> about 3000 clean comments if available
LABEL_PER_WEEK = 25           # balanced labeling file across weeks
OVERLAP_LABEL_N = 100         # overlap set for two annotators

# Request settings
COMMENT_PAGE_LIMIT = 100
POST_PAGE_LIMIT = 100
SLEEP_SEC = 0.45
MAX_RETRIES = 5
HEADERS = {"User-Agent": "Mozilla/5.0 academicRedditStudy/5.0"}

# Endpoints
BASE_COMMENTS = "https://arctic-shift.photon-reddit.com/api/comments/search"
BASE_POSTS    = "https://arctic-shift.photon-reddit.com/api/posts/search"

# Cleaning controls
MIN_TEMPLATE_WORDS = 12
AUTOMOD_RE = re.compile(r"^automoderator$", re.IGNORECASE)
MODTEAM_RE = re.compile(r"(?:^|[-_ ])modteam(?:$|[-_ ])", re.IGNORECASE)

# =========================================================
# HELPERS
# =========================================================
def to_ts(date_str: str) -> int:
    return int(datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())

def ts_to_iso(ts: int) -> str:
    return datetime.fromtimestamp(int(ts), tz=timezone.utc).isoformat()

def normalize_text(x):
    if x is None:
        return ""
    return str(x).strip()

def build_week_ranges(start_date: str, end_date: str):
    start_ts = to_ts(start_date)
    end_ts = to_ts(end_date) + 86399
    windows = []

    cursor = start_ts
    i = 1
    while cursor <= end_ts:
        win_end = min(cursor + 7 * 86400 - 1, end_ts)
        windows.append({
            "week_idx": i,
            "after_ts": int(cursor),
            "before_ts": int(win_end),
            "week_label": f"W{i:02d}_{ts_to_iso(cursor)[:10]}_to_{ts_to_iso(win_end)[:10]}",
            "window_start_utc": ts_to_iso(cursor),
            "window_end_utc": ts_to_iso(win_end),
        })
        cursor = win_end + 1
        i += 1

    return windows

def fetch_with_retry(url, params, max_retries=MAX_RETRIES):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=30)

            if r.status_code == 200:
                return r.json()

            elif r.status_code == 429:
                wait = 8 * (2 ** attempt)
                print(f"[429] Rate limited. Waiting {wait}s...")
                time.sleep(wait)

            elif r.status_code in (502, 503):
                wait = 5 * (2 ** attempt)
                print(f"[{r.status_code}] Server error. Waiting {wait}s...")
                time.sleep(wait)

            else:
                print(f"[HTTP {r.status_code}] Unrecoverable for {url}")
                print("Params:", params)
                return None

        except requests.exceptions.RequestException as e:
            wait = 5 * (2 ** attempt)
            print(f"[Connection error] {e}. Waiting {wait}s...")
            time.sleep(wait)

    print(f"[FAILED] {url} after {max_retries} attempts.")
    return None

def extract_post_id_from_link_id(link_id):
    link_id = normalize_text(link_id)
    if link_id.startswith("t3_"):
        return link_id[3:]
    return link_id

def extract_parent_comment_id(parent_id):
    parent_id = normalize_text(parent_id)
    if parent_id.startswith("t1_"):
        return parent_id[3:]
    return None

def comment_exclusion_reason(c):
    author = normalize_text(c.get("author"))
    body = normalize_text(c.get("body"))
    distinguished = normalize_text(c.get("distinguished") or c.get("distinguished_by")).lower()
    stickied = bool(c.get("stickied", False))

    if author in {"", "[deleted]"}:
        return "blank_or_deleted_author"
    if AUTOMOD_RE.match(author):
        return "automoderator"
    if MODTEAM_RE.search(author):
        return "modteam_style_username"
    if distinguished == "moderator":
        return "distinguished_moderator"
    if stickied:
        return "stickied_comment"
    if body in {"", "[removed]", "[deleted]"}:
        return "removed_or_empty_body"

    return None

def post_flags(p):
    author = normalize_text(p.get("author"))
    title = normalize_text(p.get("title"))
    selftext = normalize_text(p.get("selftext"))
    distinguished = normalize_text(p.get("distinguished") or p.get("distinguished_by")).lower()
    stickied = bool(p.get("stickied", False))

    return {
        "post_author_blank_or_deleted": int(author in {"", "[deleted]"}),
        "post_author_automod": int(bool(AUTOMOD_RE.match(author))),
        "post_author_modteam_style": int(bool(MODTEAM_RE.search(author))),
        "post_distinguished_moderator": int(distinguished == "moderator"),
        "post_stickied": int(stickied),
        "post_title_blank": int(title == ""),
        "post_body_removed": int(selftext in {"[removed]", "[deleted]"}),
        "post_body_blank": int(selftext == ""),
        "post_text_available": int(selftext not in {"", "[removed]", "[deleted]"}),
    }

def add_time_columns(df, created_col="created_utc"):
    df = df.copy()
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df[created_col], unit="s", utc=True)
    date_naive = df["date"].dt.tz_localize(None)
    df["week_start"] = date_naive.dt.to_period("W").apply(lambda r: r.start_time)
    df["year_week"] = df["week_start"].dt.strftime("%Y-W%U")
    df["parent_comment_id"] = df["parent_id"].apply(extract_parent_comment_id)
    return df

def normalize_for_template_check(text):
    text = normalize_text(text).lower()
    text = re.sub(r'https?://\S+', '<URL>', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def word_count(text):
    return len(normalize_text(text).split())

# =========================================================
# WEEK-BALANCED COMMENT COLLECTION
# =========================================================
def collect_clean_comments_for_window(subreddit, after_ts, before_ts, week_idx, week_label, target_clean):
    raw_rows = []
    clean_rows = []
    exclusion_rows = []
    seen_comment_ids = set()
    cursor = int(after_ts)

    pbar = tqdm(total=target_clean, desc=f"collect {week_label}", leave=False)

    while cursor <= before_ts:
        params = {
            "subreddit": subreddit,
            "after": int(cursor),
            "before": int(before_ts),
            "limit": COMMENT_PAGE_LIMIT,
            "sort": "asc",
        }

        data = fetch_with_retry(BASE_COMMENTS, params)
        if not data:
            break

        batch = data.get("data", [])
        if not batch:
            break

        unique_batch = []
        for c in batch:
            cid = c.get("id")
            if not cid or cid in seen_comment_ids:
                continue
            seen_comment_ids.add(cid)
            unique_batch.append(c)

        if not unique_batch:
            cursor += 1
            continue

        max_created = cursor

        for c in unique_batch:
            created_utc = int(c.get("created_utc", 0))
            max_created = max(max_created, created_utc)

            row = {
                "comment_id": c.get("id"),
                "post_id": extract_post_id_from_link_id(c.get("link_id")),
                "subreddit": c.get("subreddit"),
                "parent_id": c.get("parent_id"),
                "author": c.get("author"),
                "created_utc": created_utc,
                "text": c.get("body"),
                "distinguished": c.get("distinguished") or c.get("distinguished_by"),
                "stickied": bool(c.get("stickied", False)),
                "collection_week_idx": week_idx,
                "collection_week_label": week_label,
                "pulled_at_utc": PULLED_AT_UTC,
            }
            raw_rows.append(row)

            reason = comment_exclusion_reason(c)
            if reason is None:
                clean_rows.append(row.copy())
            else:
                exclusion_rows.append({
                    "comment_id": row["comment_id"],
                    "post_id": row["post_id"],
                    "author": row["author"],
                    "created_utc": row["created_utc"],
                    "reason": reason,
                    "body_preview": normalize_text(row["text"])[:220],
                    "collection_week_idx": week_idx,
                    "collection_week_label": week_label,
                    "pulled_at_utc": PULLED_AT_UTC,
                })

        pbar.n = min(len(clean_rows), target_clean)
        pbar.refresh()

        if len(clean_rows) >= target_clean:
            break

        cursor = max_created + 1
        time.sleep(SLEEP_SEC)

    pbar.close()

    raw_df = pd.DataFrame(raw_rows).drop_duplicates(subset=["comment_id"]).reset_index(drop=True)
    clean_df = pd.DataFrame(clean_rows).drop_duplicates(subset=["comment_id"]).reset_index(drop=True)
    excl_df = pd.DataFrame(exclusion_rows).drop_duplicates(subset=["comment_id"]).reset_index(drop=True)

    return raw_df, clean_df, excl_df

def remove_repeated_templates(comments_df):
    if comments_df.empty:
        empty_removed = pd.DataFrame(columns=list(comments_df.columns) + ["template_reason"])
        return comments_df.copy(), empty_removed

    tmp = comments_df.copy().sort_values(["author", "created_utc", "comment_id"]).reset_index(drop=True)
    tmp["template_norm"] = tmp["text"].apply(normalize_for_template_check)
    tmp["template_word_count"] = tmp["text"].apply(word_count)

    grp = (
        tmp.groupby(["author", "template_norm"])
           .agg(n_rows=("comment_id", "size"),
                n_posts=("post_id", "nunique"))
           .reset_index()
    )

    tmp = tmp.merge(grp, on=["author", "template_norm"], how="left")
    tmp["template_dup_rank"] = tmp.groupby(["author", "template_norm"]).cumcount()

    remove_mask = (
        (tmp["template_word_count"] >= MIN_TEMPLATE_WORDS) &
        (tmp["n_rows"] >= 2) &
        (tmp["n_posts"] >= 2) &
        (tmp["template_dup_rank"] >= 1)
    )

    removed_df = tmp[remove_mask].copy()
    removed_df["template_reason"] = "repeated_template_same_author"

    kept_df = tmp[~remove_mask].copy()

    drop_cols = ["template_norm", "template_word_count", "n_rows", "n_posts", "template_dup_rank"]
    kept_df = kept_df.drop(columns=drop_cols).reset_index(drop=True)
    removed_df = removed_df.drop(columns=drop_cols).reset_index(drop=True)

    return kept_df, removed_df

def collect_anchor_posts_across_window(subreddit, week_ranges, wanted_ids):
    found_posts = []
    seen_post_ids = set()

    with tqdm(total=len(wanted_ids), desc="anchor posts found") as pbar:
        for win in week_ranges:
            if len(seen_post_ids) >= len(wanted_ids):
                break

            cursor = int(win["after_ts"])
            before_ts = int(win["before_ts"])

            while cursor <= before_ts and len(seen_post_ids) < len(wanted_ids):
                params = {
                    "subreddit": subreddit,
                    "after": int(cursor),
                    "before": int(before_ts),
                    "limit": POST_PAGE_LIMIT,
                    "sort": "asc",
                }

                data = fetch_with_retry(BASE_POSTS, params)
                if not data:
                    break

                batch = data.get("data", [])
                if not batch:
                    break

                max_created = cursor

                for p in batch:
                    pid = normalize_text(p.get("id"))
                    created_utc = int(p.get("created_utc", 0))
                    max_created = max(max_created, created_utc)

                    if pid in wanted_ids and pid not in seen_post_ids:
                        row = {
                            "post_id": pid,
                            "subreddit": p.get("subreddit"),
                            "post_author": p.get("author"),
                            "post_created_utc": created_utc,
                            "post_title": p.get("title"),
                            "post_text": p.get("selftext"),
                            "pulled_at_utc": PULLED_AT_UTC,
                        }
                        row.update(post_flags(p))
                        found_posts.append(row)
                        seen_post_ids.add(pid)
                        pbar.update(1)

                cursor = max_created + 1
                time.sleep(SLEEP_SEC)

    return pd.DataFrame(found_posts).drop_duplicates(subset=["post_id"]).reset_index(drop=True)

def prune_broken_comment_links(comments_df, posts_df):
    comments_df = comments_df.copy().reset_index(drop=True)
    kept_post_ids = set(posts_df["post_id"].tolist())

    removed_total = 0

    while True:
        kept_comment_ids = set(comments_df["comment_id"].tolist())

        def parent_exists(parent_id):
            parent_id = normalize_text(parent_id)
            if parent_id.startswith("t3_"):
                return parent_id[3:] in kept_post_ids
            if parent_id.startswith("t1_"):
                return parent_id[3:] in kept_comment_ids
            return False

        keep_mask = comments_df["parent_id"].apply(parent_exists)
        removed_now = int((~keep_mask).sum())

        if removed_now == 0:
            return comments_df.reset_index(drop=True), removed_total

        removed_total += removed_now
        comments_df = comments_df[keep_mask].reset_index(drop=True)

def make_balanced_label_file(comments_df, per_week=LABEL_PER_WEEK):
    if comments_df.empty:
        return comments_df.copy()

    parts = []
    for yw, grp in comments_df.groupby("year_week", sort=True):
        n = min(per_week, len(grp))
        parts.append(grp.sample(n=n, random_state=42))

    out = pd.concat(parts, ignore_index=True)
    out = out.sort_values(["date", "comment_id"]).reset_index(drop=True)

    return out[[
        "comment_id", "post_id", "subreddit", "year_week", "author", "text"
    ]]

def make_overlap_file(comments_df, n=OVERLAP_LABEL_N):
    if comments_df.empty:
        return comments_df.copy()
    n = min(n, len(comments_df))
    out = comments_df.sample(n=n, random_state=7).sort_values(["date", "comment_id"]).reset_index(drop=True)
    return out[[
        "comment_id", "post_id", "subreddit", "year_week", "author", "text"
    ]]

# =========================================================
# RUN
# =========================================================
START_TS = to_ts(START_DATE)
END_TS = to_ts(END_DATE) + 86399
PULLED_AT_UTC = datetime.now(timezone.utc).isoformat()
week_ranges = build_week_ranges(START_DATE, END_DATE)

print(f"Subreddit      : r/{SUBREDDIT}")
print(f"Window         : {START_DATE} -> {END_DATE}")
print(f"Unix range     : {START_TS} -> {END_TS}")
print(f"Pulled at UTC  : {PULLED_AT_UTC}")
print(f"Week windows   : {len(week_ranges)}")
print(f"Target/week    : {TARGET_CLEAN_PER_WEEK}")

# Smoke test
smoke = fetch_with_retry(BASE_COMMENTS, {
    "subreddit": SUBREDDIT,
    "after": START_TS,
    "before": min(START_TS + 86400, END_TS),
    "limit": 5,
    "sort": "asc",
})
if smoke is None:
    raise RuntimeError("Comment endpoint smoke test failed.")
print(f"Comment endpoint smoke test succeeded. Returned {len(smoke.get('data', []))} rows.\n")

# 1) Collect comments week by week
raw_parts = []
clean_parts = []
excl_parts = []

for win in week_ranges:
    raw_df, clean_df, excl_df = collect_clean_comments_for_window(
        SUBREDDIT,
        win["after_ts"],
        win["before_ts"],
        win["week_idx"],
        win["week_label"],
        TARGET_CLEAN_PER_WEEK,
    )
    raw_parts.append(raw_df)
    clean_parts.append(clean_df)
    excl_parts.append(excl_df)

comments_raw_df = pd.concat(raw_parts, ignore_index=True) if raw_parts else pd.DataFrame()
comments_clean_df = pd.concat(clean_parts, ignore_index=True) if clean_parts else pd.DataFrame()
comment_exclusions_df = pd.concat(excl_parts, ignore_index=True) if excl_parts else pd.DataFrame()

comments_raw_df = comments_raw_df.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)
comments_clean_df = comments_clean_df.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)
comment_exclusions_df = comment_exclusions_df.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)

print("Raw comments collected   :", len(comments_raw_df))
print("Clean before templates   :", len(comments_clean_df))
print("Excluded moderation/sys  :", len(comment_exclusions_df))

# 2) Add time/reply columns
comments_clean_df = add_time_columns(comments_clean_df)

# 3) Remove repeated same-author boilerplate
comments_retained_df, template_removed_df = remove_repeated_templates(comments_clean_df)

print("Removed repeated templates:", len(template_removed_df))
print("Clean after templates     :", len(comments_retained_df))

# 4) Fetch anchor posts only for retained comments
wanted_post_ids = set(comments_retained_df["post_id"].dropna().astype(str).tolist())
posts_anchor_df = collect_anchor_posts_across_window(SUBREDDIT, week_ranges, wanted_post_ids)

print("Wanted anchor posts       :", len(wanted_post_ids))
print("Found anchor posts        :", len(posts_anchor_df))

# 5) Prune comments with broken parent links after final retention
comments_final_df, pruned_n = prune_broken_comment_links(comments_retained_df, posts_anchor_df)
comments_final_df = add_time_columns(comments_final_df)

final_comment_ids = set(comments_final_df["comment_id"].tolist())
comments_final_df["has_parent_in_window"] = comments_final_df["parent_comment_id"].isin(final_comment_ids)
comments_final_df["is_direct_reply"] = comments_final_df["parent_id"].astype(str).str.startswith("t3_")
comments_final_df["parent_post_found"] = comments_final_df["post_id"].isin(set(posts_anchor_df["post_id"].tolist()))

print("Pruned broken-link comments:", pruned_n)
print("Final clean comments        :", len(comments_final_df))

# 6) Balanced labeling outputs
to_label_df = make_balanced_label_file(comments_final_df, per_week=LABEL_PER_WEEK)
overlap_label_df = make_overlap_file(comments_final_df, n=OVERLAP_LABEL_N)

# 7) Weekly summary
def count_by_week(df, label_col="collection_week_label", value_name="count"):
    if df.empty or label_col not in df.columns:
        return pd.DataFrame(columns=[label_col, value_name])
    return df.groupby(label_col).size().reset_index(name=value_name)

weekly_summary_df = pd.DataFrame({"collection_week_label": [w["week_label"] for w in week_ranges]})

weekly_summary_df = weekly_summary_df.merge(
    count_by_week(comments_raw_df, value_name="raw_comments"),
    on="collection_week_label", how="left"
)
weekly_summary_df = weekly_summary_df.merge(
    count_by_week(comments_clean_df, value_name="clean_before_templates"),
    on="collection_week_label", how="left"
)
weekly_summary_df = weekly_summary_df.merge(
    count_by_week(comment_exclusions_df, value_name="excluded_mod_sys"),
    on="collection_week_label", how="left"
)
weekly_summary_df = weekly_summary_df.merge(
    count_by_week(template_removed_df, value_name="removed_templates"),
    on="collection_week_label", how="left"
)
weekly_summary_df = weekly_summary_df.merge(
    count_by_week(comments_final_df, value_name="final_clean_comments"),
    on="collection_week_label", how="left"
)

weekly_summary_df = weekly_summary_df.fillna(0)

# 8) Save outputs
TAG = f"{START_DATE.replace('-', '')}_{END_DATE.replace('-', '')}_weekbalanced"

comments_raw_out        = f"{SUBREDDIT}_comments_raw_{TAG}.csv"
comments_clean_out      = f"{SUBREDDIT}_comments_clean_final_{TAG}.csv"
comments_excl_out       = f"{SUBREDDIT}_comment_exclusions_{TAG}.csv"
comments_template_out   = f"{SUBREDDIT}_comment_templates_removed_{TAG}.csv"
posts_anchor_out        = f"{SUBREDDIT}_anchor_posts_{TAG}.csv"
weekly_summary_out      = f"{SUBREDDIT}_weekly_summary_{TAG}.csv"
to_label_out            = f"{SUBREDDIT}_to_label_balanced_{TAG}.csv"
overlap_label_out       = f"{SUBREDDIT}_to_label_overlap_{TAG}.csv"

comments_raw_df.to_csv(comments_raw_out, index=False)
comments_final_df.to_csv(comments_clean_out, index=False)
comment_exclusions_df.to_csv(comments_excl_out, index=False)
template_removed_df.to_csv(comments_template_out, index=False)
posts_anchor_df.to_csv(posts_anchor_out, index=False)
weekly_summary_df.to_csv(weekly_summary_out, index=False)
to_label_df.to_csv(to_label_out, index=False)
overlap_label_df.to_csv(overlap_label_out, index=False)

# 9) Final summary
overall_summary_df = pd.DataFrame([
    ["subreddit", SUBREDDIT],
    ["window_start", START_DATE],
    ["window_end", END_DATE],
    ["pulled_at_utc", PULLED_AT_UTC],
    ["num_week_windows", len(week_ranges)],
    ["target_clean_per_week", TARGET_CLEAN_PER_WEEK],
    ["raw_comments", len(comments_raw_df)],
    ["clean_before_templates", len(comments_clean_df)],
    ["excluded_mod_system", len(comment_exclusions_df)],
    ["removed_repeated_templates", len(template_removed_df)],
    ["final_clean_comments", len(comments_final_df)],
    ["anchor_posts_found", len(posts_anchor_df)],
    ["unique_comment_authors", comments_final_df["author"].nunique() if not comments_final_df.empty else 0],
    ["unique_posts_in_final", comments_final_df["post_id"].nunique() if not comments_final_df.empty else 0],
    ["weeks_covered_final", comments_final_df["year_week"].nunique() if not comments_final_df.empty else 0],
    ["to_label_rows", len(to_label_df)],
    ["overlap_label_rows", len(overlap_label_df)],
], columns=["metric", "value"])

reason_counts_df = (
    comment_exclusions_df["reason"].value_counts(dropna=False)
    .rename_axis("reason")
    .reset_index(name="count")
    if not comment_exclusions_df.empty else pd.DataFrame(columns=["reason", "count"])
)

print("\n=== OVERALL SUMMARY ===")
print(overall_summary_df.to_string(index=False))

print("\n=== MOD/SYSTEM EXCLUSION REASONS ===")
print(reason_counts_df.to_string(index=False) if not reason_counts_df.empty else "No exclusions.")

print("\n=== FINAL CLEAN COMMENT DATE RANGE ===")
if not comments_final_df.empty:
    print("Min:", comments_final_df["date"].min())
    print("Max:", comments_final_df["date"].max())

print("\n=== WEEKLY COVERAGE ===")
print(weekly_summary_df.to_string(index=False))

print("\nSaved files:")
print(" -", comments_raw_out)
print(" -", comments_clean_out)
print(" -", comments_excl_out)
print(" -", comments_template_out)
print(" -", posts_anchor_out)
print(" -", weekly_summary_out)
print(" -", to_label_out)
print(" -", overlap_label_out)

# 10) Optional Colab display
try:
    from google.colab import data_table
    from IPython.display import display, Markdown
    data_table.enable_dataframe_formatter()

    display(Markdown("## Overall Summary"))
    display(overall_summary_df)

    display(Markdown("## Weekly Summary"))
    display(weekly_summary_df)

    display(Markdown("## Exclusion Reasons"))
    display(reason_counts_df)

    display(Markdown("## Final Clean Comments Preview"))
    display(comments_final_df[[
        "comment_id", "post_id", "author", "date", "year_week",
        "parent_id", "parent_comment_id", "has_parent_in_window",
        "is_direct_reply", "text"
    ]].head(50))

    display(Markdown("## Removed Repeated Templates Preview"))
    if not template_removed_df.empty:
        display(template_removed_df[[
            "comment_id", "post_id", "author", "date", "year_week",
            "template_reason", "text"
        ]].head(50))

    display(Markdown("## Anchor Posts Preview"))
    if not posts_anchor_df.empty:
        display(posts_anchor_df[[
            "post_id", "post_author", "post_created_utc", "post_title",
            "post_text_available", "post_body_removed", "post_stickied"
        ]].head(50))

except Exception as e:
    print("\nInteractive Colab display unavailable:", e)

Subreddit      : r/askscience
Window         : 2025-10-06 -> 2025-12-28
Unix range     : 1759708800 -> 1766966399
Pulled at UTC  : 2026-04-07T03:14:42.465793+00:00
Week windows   : 12
Target/week    : 250
Comment endpoint smoke test succeeded. Returned 5 rows.



collect W01_2025-10-06_to_2025-10-12:   0%|          | 0/250 [00:00<?, ?it/s]

collect W02_2025-10-13_to_2025-10-19:   0%|          | 0/250 [00:00<?, ?it/s]

collect W03_2025-10-20_to_2025-10-26:   0%|          | 0/250 [00:00<?, ?it/s]

collect W04_2025-10-27_to_2025-11-02:   0%|          | 0/250 [00:00<?, ?it/s]

...

,comment_id,post_id,author,date,year_week,parent_id,parent_comment_id,has_parent_in_window,is_direct_reply,text
0,nrrcpjy,1pb9jq8,--Chug--,2025-12-01 19:25:23+00:00,2025-W48,t1_nrqnbic,nrqnbic,True,False,I'd say they're still planets just not major o...
1,njpquxv,1o7a3uc,-Metacelsus-,2025-10-16 00:18:41+00:00,2025-W41,t1_njoras5,njoras5,True,False,> yet if we give it to certain primates it cou...
2,nu55nbz,1pmpuzp,-Metacelsus-,2025-12-15 12:04:57+00:00,2025-W50,t1_nu4ygzy,nu4ygzy,True,False,> This bounces light through the crystalline l...
3,nicxdqm,1nzk1tr,-dantes-,2025-10-08 02:41:01+00:00,2025-W40,t1_ni4ivj4,ni4ivj4,True,False,I heard somewhere that creatures that evolved ...
4,njmxjfw,1o4ves5,095179005,2025-10-15 15:30:35+00:00,2025-W41,t3_1o4ves5,None,False,True,"Aside from the wonderful detailed answers,\n\n..."
5,nkg7g72,1oag8fc,095179005,2025-10-20 08:57:52+00:00,2025-W42,t3_1oag8fc,None,False,True,An addition to the explanations and reasons ot...
6,nkulx98,1od93jq,095179005,2025-10-22 21:18:33+00:00,2025-W42,t1_nktkm9o,nktkm9o,True,False,This response about how cells know where they ...
7,nqh9yef,1p4o0yp,095179005,2025-11-24 04:35:45+00:00,2025-W47,t3_1p4o0yp,None,False,True,Evolution is descent with modification.\n\nAll...
8,nrn10rj,1pareq7,095179005,2025-12-01 01:48:25+00:00,2025-W48,t3_1pareq7,None,False,True,So the paper states that out of 160 known spec...
9,nvp0ssh,1pu20gm,0oSlytho0,2025-12-24 09:54:24+00:00,2025-W51,t1_nvmo8jm,nvmo8jm,True,False,"That can also be a happy coincident, and a sli..."


## Removed Repeated Templates Preview

,comment_id,post_id,author,date,year_week,template_reason,text
0,ni0raik,1nzaihs,stars9r9in9the9past,2025-10-06 05:19:22+00:00,2025-W40,repeated_template_same_author,I was look at the upcoming 7-day for the week ...
1,ni0rhku,1nzajlx,stars9r9in9the9past,2025-10-06 05:21:06+00:00,2025-W40,repeated_template_same_author,I was look at the upcoming 7-day for the week ...


## Anchor Posts Preview

,post_id,post_author,post_created_utc,post_title,post_text_available,post_body_removed,post_stickied
0,1nz6p4c,AirportStraight8079,1759716043,Do the strongest earthquake permanently rise g...,1,0,0
1,1nzagds,stars9r9in9the9past,1759727747,Basics to Meteorology,0,1,0
2,1nzk1tr,Derole,1759759652,Why are we not crushed by the air above us?,1,0,0
3,1o09tql,disconcertedad1023,1759828571,Why are airplane windows not as large as helic...,0,0,0
4,1o0m74r,L1fel0ver2002,1759860076,Why are stars/planets only spherical?,0,1,0
5,1o0ptjd,Electronic_Click_470,1759867937,Could information travel faster than light thr...,0,1,0
6,1o1wkod,LikesBlueberriesALot,1759983578,"It seems like birds, rabbits, squirrels etc. w...",0,0,0
7,1o1zcqf,mooman996,1759993807,Is Earth getting smoother over time?,1,0,0
8,1o250a0,jemmylegs,1760014088,Why do we need red blood cells?,1,0,0
9,1o25xvb,AkelaAnda,1760016501,"if anything with mass curves spacetime, even i...",1,0,0
